# Avant le TP3 : faire tourner des consommateurs dans le notebook

La version Markdown du TP3 ouvre trois terminaux pour faire tourner plusieurs consommateurs d'un même groupe. Dans le notebook, chaque consommateur est une **instance** qui tourne en arrière-plan. Cette démonstration montre comment on les lance, les observe et les arrête, sur un topic jetable `demo-tp3` de 4 partitions :

1. une instance seule reçoit toutes les partitions ;
2. une seconde instance déclenche un rééquilibrage ;
3. le groupe se décrit en Python comme en ligne de commande ;
4. une instance arrêtée proprement rend ses partitions aussitôt.

Sorties de référence : exécution du 09/11/2026 23:04 UTC sur le lab.

In [1]:
import os
import threading
import time

from confluent_kafka import Consumer
from confluent_kafka.admin import AdminClient, NewTopic

BS = os.environ["BS"]
admin = AdminClient({"bootstrap.servers": BS})

admin.create_topics([NewTopic("demo-tp3", num_partitions=4, replication_factor=3)])["demo-tp3"].result()
print("demo-tp3 cree")

demo-tp3 cree


## La classe `Instance`

C'est la même que dans le TP3. Un `Consumer` abonné au topic tourne dans un fil d'exécution : sa boucle `consume()` fait vivre son appartenance au groupe, et `affectation` donne ses partitions du moment.

In [2]:
class Instance:
    """Une instance d'un service consommateur : un Consumer qui tourne dans un fil d'execution."""

    lancees = []

    def __init__(self, groupe, topic="orders", **config):
        self.groupe = groupe
        self.affectation = []
        self._arret = threading.Event()
        self.consommateur = Consumer({"bootstrap.servers": BS, "group.id": groupe,
                                      "auto.offset.reset": "earliest", "log_level": 4, **config})
        self.consommateur.subscribe([topic])
        self._fil = threading.Thread(target=self._boucle, daemon=True)
        self._fil.start()
        Instance.lancees.append(self)

    def _boucle(self):
        while not self._arret.is_set():
            # consume() fait aussi vivre l'appartenance au groupe (reequilibrages, validations)
            self.consommateur.consume(100, timeout=0.5)
            self.affectation = sorted(tp.partition for tp in self.consommateur.assignment())
        # Depart propre : le coordinateur est prevenu et reequilibre aussitot
        self.consommateur.close()

    def arreter(self):
        self._arret.set()
        self._fil.join(15)

    @classmethod
    def arreter_tout(cls):
        for instance in cls.lancees:
            instance.arreter()
        cls.lancees = []

    def __repr__(self):
        return "Instance(%s, partitions %s)" % (self.groupe, self.affectation)

## 1. Une instance seule

In [3]:
x = Instance("demo-tp3", topic="demo-tp3")
while not x.affectation:
    time.sleep(1)
print(x)

Instance(demo-tp3, partitions [0, 1, 2, 3])


## 2. Une seconde instance : rééquilibrage

Une boucle d'attente rappelle la condition jusqu'à ce qu'elle soit vraie : ici, jusqu'à ce que les deux instances aient chacune des partitions.

In [4]:
y = Instance("demo-tp3", topic="demo-tp3")
while not (x.affectation and y.affectation):
    time.sleep(1)
print(x)
print(y)

Instance(demo-tp3, partitions [0, 1])
Instance(demo-tp3, partitions [2, 3])


## 3. Décrire le groupe

En Python, `describe_consumer_groups()` renvoie l'état, la stratégie d'affectation et les membres. L'outil Java donne la même vue.

In [5]:
groupe = admin.describe_consumer_groups(["demo-tp3"])["demo-tp3"].result()
print(groupe.state.name, groupe.type.name, groupe.partition_assignor)
for m in groupe.members:
    print(m.member_id, sorted(tp.partition for tp in m.assignment.topic_partitions))

STABLE CLASSIC range
rdkafka-4d715772-072f-46f8-a35d-ada8f4c0a387 [2, 3]
rdkafka-1eb5d0a8-c55a-4c07-a03f-1b2b585d0f76 [0, 1]


In [6]:
!docker exec kafka-tools kafka-consumer-groups.sh --bootstrap-server $BS --describe --group demo-tp3 --state


GROUP           COORDINATOR (ID)          ASSIGNMENT-STRATEGY  STATE                #MEMBERS
demo-tp3        kafka-1:19092  (1)        range                Stable               2


## 4. Arrêt propre d'une instance

In [7]:
y.arreter()
while len(x.affectation) != 4:
    time.sleep(1)
print(x)

Instance(demo-tp3, partitions [0, 1, 2, 3])


## Ménage

`Instance.arreter_tout()` arrête toutes les instances lancées : à utiliser en cas de doute, par exemple après avoir exécuté deux fois une cellule qui en crée. Le groupe et le topic de démonstration sont ensuite supprimés.

In [8]:
Instance.arreter_tout()
admin.delete_consumer_groups(["demo-tp3"])["demo-tp3"].result()
admin.delete_topics(["demo-tp3"])["demo-tp3"].result()
print("groupes :", sorted(g.group_id for g in admin.list_consumer_groups().result().valid))

groupes : ['console-consumer-15711', 'console-consumer-44511', 'loadgen-consumers']


## La méthode du TP3

Ouvrez `tps/jour1/tp3_groupes_retention/enonce.ipynb`.

- Les consommateurs du TP sont des instances : lancez-les, attendez leur affectation avec une boucle d'attente, arrêtez-les avec `arreter()`.
- Une cellule qui crée des instances ne s'exécute qu'une fois ; en cas de doute, `Instance.arreter_tout()`.
- Les étapes de rétention et de compaction attendent le travail du broker : une à deux minutes par étape, pendant lesquelles la cellule reste en cours d'exécution (`[*]`).